<a href="https://colab.research.google.com/github/Anshulmohan27/Machine_Learning/blob/main/Feature_Engineering_of_Stock_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [4]:
stock_data = pd.read_csv('nifty500_5yr_data.csv')
stock_data.head()

,Date,Ticker,Open,High,Low,Close,Volume,Notes
0,2021-06-09,360ONE,260.813755,262.150806,249.887157,251.816010,59152.0,NaN
1,2021-06-10,360ONE,253.997012,260.046642,251.848943,257.131409,41084.0,NaN
2,2021-06-11,360ONE,259.443804,263.246739,256.714877,260.758942,232800.0,NaN
3,2021-06-14,360ONE,261.931602,263.027550,258.939658,259.619141,74364.0,NaN
4,2021-06-15,360ONE,260.791773,260.791773,252.144728,253.821533,149636.0,NaN


In [5]:
stock_data.isnull().sum()/stock_data.shape[0]

,0
Date,0.000000
Ticker,0.000000
Open,0.000000
High,0.000000
Low,0.000000
Close,0.000000
Volume,0.000000
Notes,0.999157


In [6]:
stock_data = stock_data.drop(columns = 'Notes', axis = 1)

In [7]:
stock_data.head()

,Date,Ticker,Open,High,Low,Close,Volume
0,2021-06-09,360ONE,260.813755,262.150806,249.887157,251.816010,59152.0
1,2021-06-10,360ONE,253.997012,260.046642,251.848943,257.131409,41084.0
2,2021-06-11,360ONE,259.443804,263.246739,256.714877,260.758942,232800.0
3,2021-06-14,360ONE,261.931602,263.027550,258.939658,259.619141,74364.0
4,2021-06-15,360ONE,260.791773,260.791773,252.144728,253.821533,149636.0


In [8]:
stock_data['Ticker'].nunique()/stock_data.shape[0]

0.00090638329499332

In [9]:
df_stock_cat = stock_data.groupby('Ticker').size()

In [10]:
df_stock_cat.rename({"0": "Num"}, inplace=True)
print(df_stock_cat)

Ticker
360ONE        1233
3MINDIA       1233
AADHARHFC      510
AARTIIND      1234
AAVAS         1234
              ... 
ZENSARTECH    1234
ZENTEC        1234
ZFCVINDIA     1233
ZYDUSLIFE     1230
ZYDUSWELL     1234
Length: 500, dtype: int64


In [11]:
ticker_dataframes = {}
for ticker in stock_data['Ticker'].unique():
    ticker_dataframes[ticker] = stock_data[stock_data['Ticker'] == ticker]

# To verify, you can check one of the created dataframes, for example:
print(ticker_dataframes['360ONE'].head())

         Date  Ticker        Open        High         Low       Close  \
0  2021-06-09  360ONE  260.813755  262.150806  249.887157  251.816010   
1  2021-06-10  360ONE  253.997012  260.046642  251.848943  257.131409   
2  2021-06-11  360ONE  259.443804  263.246739  256.714877  260.758942   
3  2021-06-14  360ONE  261.931602  263.027550  258.939658  259.619141   
4  2021-06-15  360ONE  260.791773  260.791773  252.144728  253.821533   

     Volume  
0   59152.0  
1   41084.0  
2  232800.0  
3   74364.0  
4  149636.0  


In [13]:
print(ticker_dataframes['ABB'].head())

            Date Ticker         Open         High          Low        Close  \
2466  2021-06-09    ABB  1654.694704  1682.906093  1628.145639  1637.777588   
2467  2021-06-10    ABB  1642.813661  1652.592314  1623.256356  1634.355103   
2468  2021-06-11    ABB  1646.822977  1711.313267  1608.588467  1612.059937   
2469  2021-06-14    ABB  1622.278335  1635.283990  1565.562155  1623.842896   
2470  2021-06-15    ABB  1623.843171  1660.415358  1623.794349  1656.454956   

        Volume  
2466  224477.0  
2467   87950.0  
2468  124172.0  
2469  124292.0  
2470  103988.0  


In [14]:
X_dataframes = {}
y_dataframes = {}

features = ['Date', 'Ticker', 'Open', 'High', 'Low', 'Volume']
target = 'Close'

for ticker, df in ticker_dataframes.items():
    X_dataframes[ticker] = df[features]
    y_dataframes[ticker] = df[target]

# You can check the first few rows of X and y for a specific ticker, for example '360ONE':
print("X for 360ONE (first 5 rows):")
print(X_dataframes['360ONE'].head())
print("\ny for 360ONE (first 5 rows):")
print(y_dataframes['360ONE'].head())

X for 360ONE (first 5 rows):
         Date  Ticker        Open        High         Low    Volume
0  2021-06-09  360ONE  260.813755  262.150806  249.887157   59152.0
1  2021-06-10  360ONE  253.997012  260.046642  251.848943   41084.0
2  2021-06-11  360ONE  259.443804  263.246739  256.714877  232800.0
3  2021-06-14  360ONE  261.931602  263.027550  258.939658   74364.0
4  2021-06-15  360ONE  260.791773  260.791773  252.144728  149636.0

y for 360ONE (first 5 rows):
0    251.816010
1    257.131409
2    260.758942
3    259.619141
4    253.821533
Name: Close, dtype: float64


In [15]:
from sklearn.model_selection import train_test_split

X_train_dataframes = {}
X_test_dataframes = {}
y_train_dataframes = {}
y_test_dataframes = {}

# Assuming a 80/20 split for training and testing
test_size = 0.2
random_state = 42 # for reproducibility

for ticker in ticker_dataframes.keys():
    X_ticker = X_dataframes[ticker]
    y_ticker = y_dataframes[ticker]

    X_train, X_test, y_train, y_test = train_test_split(
        X_ticker, y_ticker, test_size=test_size, random_state=random_state, shuffle=False # Shuffle=False for time series data
    )

    X_train_dataframes[ticker] = X_train
    X_test_dataframes[ticker] = X_test
    y_train_dataframes[ticker] = y_train
    y_test_dataframes[ticker] = y_test

print(f"Created training and testing sets for {len(ticker_dataframes)} tickers.")

# You can verify the shapes for a specific ticker, for example '360ONE':
print("\nShape of X_train for 360ONE:", X_train_dataframes['360ONE'].shape)
print("Shape of X_test for 360ONE:", X_test_dataframes['360ONE'].shape)
print("Shape of y_train for 360ONE:", y_train_dataframes['360ONE'].shape)
print("Shape of y_test for 360ONE:", y_test_dataframes['360ONE'].shape)

Created training and testing sets for 500 tickers.

Shape of X_train for 360ONE: (986, 6)
Shape of X_test for 360ONE: (247, 6)
Shape of y_train for 360ONE: (986,)
Shape of y_test for 360ONE: (247,)


In [16]:
def feature_engineer_date(dataframe_dict):
    processed_dataframes = {}
    for ticker, df in dataframe_dict.items():
        df_copy = df.copy() # Work on a copy to avoid SettingWithCopyWarning
        df_copy['Date'] = pd.to_datetime(df_copy['Date'])
        df_copy['Year'] = df_copy['Date'].dt.year
        df_copy['Month'] = df_copy['Date'].dt.month
        df_copy['Day'] = df_copy['Date'].dt.day
        df_copy = df_copy.drop(columns=['Date'])
        processed_dataframes[ticker] = df_copy
    return processed_dataframes

X_train_dataframes = feature_engineer_date(X_train_dataframes)
X_test_dataframes = feature_engineer_date(X_test_dataframes)

print("Date feature engineering complete for all X_train and X_test dataframes.")

# Verify the changes for a specific ticker, for example '360ONE'
print("\nX_train for 360ONE (first 5 rows) after feature engineering:")
print(X_train_dataframes['360ONE'].head())
print("\nX_test for 360ONE (first 5 rows) after feature engineering:")
print(X_test_dataframes['360ONE'].head())

Date feature engineering complete for all X_train and X_test dataframes.

X_train for 360ONE (first 5 rows) after feature engineering:
   Ticker        Open        High         Low    Volume  Year  Month  Day
0  360ONE  260.813755  262.150806  249.887157   59152.0  2021      6    9
1  360ONE  253.997012  260.046642  251.848943   41084.0  2021      6   10
2  360ONE  259.443804  263.246739  256.714877  232800.0  2021      6   11
3  360ONE  261.931602  263.027550  258.939658   74364.0  2021      6   14
4  360ONE  260.791773  260.791773  252.144728  149636.0  2021      6   15

X_test for 360ONE (first 5 rows) after feature engineering:
     Ticker         Open         High          Low     Volume  Year  Month  \
986  360ONE  1065.563411  1075.603789  1035.887488  1548191.0  2025      6   
987  360ONE  1062.892623  1066.107515   987.219021   479653.0  2025      6   
988  360ONE  1061.458295  1101.768088  1048.994311  1488940.0  2025      6   
989  360ONE  1078.225214  1098.009162  1058.4907

In [17]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Dictionaries to store models, predictions, and evaluation metrics for each ticker
rf_models = {}
rf_predictions = {}
rf_mse_scores = {}

for ticker in ticker_dataframes.keys():
    print(f"\nTraining Random Forest for {ticker}...")

    # Get the training and testing data for the current ticker
    X_train = X_train_dataframes[ticker].drop(columns=['Ticker'])
    y_train = y_train_dataframes[ticker]
    X_test = X_test_dataframes[ticker].drop(columns=['Ticker'])
    y_test = y_test_dataframes[ticker]

    # Initialize and train the Random Forest Regressor model
    # You can adjust hyperparameters like n_estimators, max_depth, etc.
    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)

    # Store the trained model
    rf_models[ticker] = model

    # Make predictions on the test set
    y_pred = model.predict(X_test)

    # Store the predictions
    rf_predictions[ticker] = y_pred

    # Calculate Mean Squared Error
    mse = mean_squared_error(y_test, y_pred)
    rf_mse_scores[ticker] = mse

    print(f"Mean Squared Error for {ticker}: {mse:.2f}")

print("\nRandom Forest models trained and evaluated for all tickers.")

# Optionally, display a summary of MSE scores
print("\nSummary of Mean Squared Errors:")
for ticker, mse in rf_mse_scores.items():
    print(f"  {ticker}: {mse:.2f}")


Training Random Forest for 360ONE...
Mean Squared Error for 360ONE: 145.32

Training Random Forest for 3MINDIA...
Mean Squared Error for 3MINDIA: 89707.63

Training Random Forest for ABB...
Mean Squared Error for ABB: 3141.81

Training Random Forest for ACC...
Mean Squared Error for ACC: 15346.30

Training Random Forest for ACMESOLAR...
Mean Squared Error for ACMESOLAR: 92.98

Training Random Forest for AIAENG...
Mean Squared Error for AIAENG: 1234.22

Training Random Forest for APLAPOLLO...
Mean Squared Error for APLAPOLLO: 12019.48

Training Random Forest for AUBANK...
Mean Squared Error for AUBANK: 20176.79

Training Random Forest for AWL...
Mean Squared Error for AWL: 978.37

Training Random Forest for AADHARHFC...
Mean Squared Error for AADHARHFC: 20.33

Training Random Forest for AARTIIND...
Mean Squared Error for AARTIIND: 23.01

Training Random Forest for AAVAS...
Mean Squared Error for AAVAS: 3571.71

Training Random Forest for ABBOTINDIA...
Mean Squared Error for ABBOTINDIA:

In [18]:
from sklearn.metrics import mean_absolute_error

rf_mae_scores = {}

print("\nCalculating Mean Absolute Error for each ticker...")

for ticker in ticker_dataframes.keys():
    y_test = y_test_dataframes[ticker]
    y_pred = rf_predictions[ticker]

    mae = mean_absolute_error(y_test, y_pred)
    rf_mae_scores[ticker] = mae
    print(f"Mean Absolute Error for {ticker}: {mae:.2f}")

print("\nMean Absolute Errors calculated for all tickers.")

# Optionally, display a summary of MAE scores
print("\nSummary of Mean Absolute Errors:")
for ticker, mae in rf_mae_scores.items():
    print(f"  {ticker}: {mae:.2f}")


Calculating Mean Absolute Error for each ticker...
Mean Absolute Error for 360ONE: 9.13
Mean Absolute Error for 3MINDIA: 214.25
Mean Absolute Error for ABB: 42.29
Mean Absolute Error for ACC: 68.82
Mean Absolute Error for ACMESOLAR: 4.94
Mean Absolute Error for AIAENG: 26.48
Mean Absolute Error for APLAPOLLO: 57.63
Mean Absolute Error for AUBANK: 111.12
Mean Absolute Error for AWL: 21.35
Mean Absolute Error for AADHARHFC: 3.46
Mean Absolute Error for AARTIIND: 3.69
Mean Absolute Error for AAVAS: 30.29
Mean Absolute Error for ABBOTINDIA: 607.93
Mean Absolute Error for ACE: 9.63
Mean Absolute Error for ACUTAAS: 587.04
Mean Absolute Error for ADANIENSOL: 9.52
Mean Absolute Error for ADANIENT: 18.03
Mean Absolute Error for ADANIGREEN: 10.48
Mean Absolute Error for ADANIPORTS: 31.43
Mean Absolute Error for ADANIPOWER: 10.76
Mean Absolute Error for ATGL: 7.82
Mean Absolute Error for ABCAPITAL: 76.29
Mean Absolute Error for ABFRL: 6.78
Mean Absolute Error for ABLBL: 1.15
Mean Absolute Error 